In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
ROOT_DIR = Path("..")

RAW_DATA = ROOT_DIR / "data" / "raw"
PROCESSED_DATA = ROOT_DIR / "data" / "processed"

PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

In [3]:
def load_data():

    train = pd.read_csv(
        RAW_DATA / "train.csv",
        low_memory=False
    )

    test = pd.read_csv(
        RAW_DATA / "test.csv",
        low_memory=False
    )

    store = pd.read_csv(
        RAW_DATA / "store.csv",
        low_memory=False
    )

    superstore = pd.read_csv(
        RAW_DATA / "Superstore.csv",
        encoding="latin1",
        low_memory=False
    )

    return train, test, store, superstore


train, test, store, superstore = load_data()

In [4]:
datasets = {
    "train": train,
    "test": test,
    "store": store,
    "superstore": superstore
}

for name, df in datasets.items():

    print(f"\n{name.upper()}")
    print("-" * 50)

    print(f"Rows    : {df.shape[0]:,}")
    print(f"Columns : {df.shape[1]}")


TRAIN
--------------------------------------------------
Rows    : 1,017,209
Columns : 9

TEST
--------------------------------------------------
Rows    : 41,088
Columns : 8

STORE
--------------------------------------------------
Rows    : 1,115
Columns : 10

SUPERSTORE
--------------------------------------------------
Rows    : 9,994
Columns : 21


In [5]:
train["Date"] = pd.to_datetime(
    train["Date"],
    errors="coerce"
)

test["Date"] = pd.to_datetime(
    test["Date"],
    errors="coerce"
)

superstore["Order Date"] = pd.to_datetime(
    superstore["Order Date"],
    errors="coerce"
)

superstore["Ship Date"] = pd.to_datetime(
    superstore["Ship Date"],
    errors="coerce"
)

In [6]:
def dataset_summary(df):

    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "missing": df.isnull().sum(),
        "missing_%": round(
            df.isnull().mean() * 100,
            2
        ),
        "unique": df.nunique()
    })

    return summary

In [7]:
dataset_summary(store)

,dtype,missing,missing_%,unique
Store,int64,0,0.00,1115
StoreType,str,0,0.00,4
Assortment,str,0,0.00,3
CompetitionDistance,float64,3,0.27,654
CompetitionOpenSinceMonth,float64,354,31.75,12
CompetitionOpenSinceYear,float64,354,31.75,23
Promo2,int64,0,0.00,2
Promo2SinceWeek,float64,544,48.79,24
Promo2SinceYear,float64,544,48.79,7
PromoInterval,str,544,48.79,3


In [8]:
for name, df in datasets.items():

    print(f"\n{name.upper()}")

    missing = df.isnull().sum()

    missing = missing[missing > 0]

    if len(missing) == 0:
        print("No Missing Values")

    else:
        print(missing.sort_values(ascending=False))


TRAIN
No Missing Values

TEST
Open    11
dtype: int64

STORE
Promo2SinceYear              544
Promo2SinceWeek              544
PromoInterval                544
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
CompetitionDistance            3
dtype: int64

SUPERSTORE
No Missing Values


In [9]:
for name, df in datasets.items():

    duplicates = df.duplicated().sum()

    print(
        f"{name} : {duplicates:,} duplicate rows"
    )

train : 0 duplicate rows
test : 0 duplicate rows
store : 0 duplicate rows
superstore : 0 duplicate rows


In [10]:
store[store["CompetitionDistance"].isnull()]

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
290,291,d,a,NaN,NaN,NaN,0,NaN,NaN,NaN
621,622,a,c,NaN,NaN,NaN,0,NaN,NaN,NaN
878,879,d,a,NaN,NaN,NaN,1,5.0,2013.0,"Feb,May,Aug,Nov"


In [11]:
store[
    store["Promo2"] == 0
][[
    "Store",
    "Promo2",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval"
]].head()

,Store,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,0,NaN,NaN,NaN
3,4,0,NaN,NaN,NaN
4,5,0,NaN,NaN,NaN
5,6,0,NaN,NaN,NaN
6,7,0,NaN,NaN,NaN


In [12]:
dataset_summary(train).to_csv(
    PROCESSED_DATA / "train_audit.csv"
)

dataset_summary(store).to_csv(
    PROCESSED_DATA / "store_audit.csv"
)

dataset_summary(superstore).to_csv(
    PROCESSED_DATA / "superstore_audit.csv"
)

In [13]:
dataset_summary(store)

,dtype,missing,missing_%,unique
Store,int64,0,0.00,1115
StoreType,str,0,0.00,4
Assortment,str,0,0.00,3
CompetitionDistance,float64,3,0.27,654
CompetitionOpenSinceMonth,float64,354,31.75,12
CompetitionOpenSinceYear,float64,354,31.75,23
Promo2,int64,0,0.00,2
Promo2SinceWeek,float64,544,48.79,24
Promo2SinceYear,float64,544,48.79,7
PromoInterval,str,544,48.79,3


In [14]:
store[store["CompetitionDistance"].isnull()]

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
290,291,d,a,NaN,NaN,NaN,0,NaN,NaN,NaN
621,622,a,c,NaN,NaN,NaN,0,NaN,NaN,NaN
878,879,d,a,NaN,NaN,NaN,1,5.0,2013.0,"Feb,May,Aug,Nov"


In [15]:
train_clean = train.copy()
store_clean = store.copy()
superstore_clean = superstore.copy()

In [16]:
store_clean["CompetitionDistance"] = (
    store_clean["CompetitionDistance"]
    .fillna(
        store_clean["CompetitionDistance"].median()
    )
)

In [17]:
store_clean["CompetitionOpenSinceMonth"] = (
    store_clean["CompetitionOpenSinceMonth"]
    .fillna(0)
)

store_clean["CompetitionOpenSinceYear"] = (
    store_clean["CompetitionOpenSinceYear"]
    .fillna(0)
)

In [18]:
store_clean["Promo2SinceWeek"] = (
    store_clean["Promo2SinceWeek"]
    .fillna(0)
)

store_clean["Promo2SinceYear"] = (
    store_clean["Promo2SinceYear"]
    .fillna(0)
)

store_clean["PromoInterval"] = (
    store_clean["PromoInterval"]
    .fillna("No Promo")
)

In [19]:
store_clean.isnull().sum()

Store                        0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
dtype: int64

In [20]:
train_store = train_clean.merge(
    store_clean,
    on="Store",
    how="left"
)

In [21]:
train_store.shape

(1017209, 18)

In [22]:
train_store["Year"] = train_store["Date"].dt.year
train_store["Month"] = train_store["Date"].dt.month
train_store["Day"] = train_store["Date"].dt.day
train_store["Week"] = train_store["Date"].dt.isocalendar().week
train_store["Quarter"] = train_store["Date"].dt.quarter

In [23]:
train_store["IsWeekend"] = (
    train_store["DayOfWeek"]
    .isin([6, 7])
    .astype(int)
)

In [24]:
train_store.to_csv(
    "../data/processed/rossmann_clean.csv",
    index=False
)

superstore_clean.to_csv(
    "../data/processed/superstore_clean.csv",
    index=False
)

Feature 1
Competition Age

In [25]:
train_store["CompetitionAge"] = (
    train_store["Year"]
    - train_store["CompetitionOpenSinceYear"]
)

train_store["CompetitionAge"] = (
    train_store["CompetitionAge"]
    .clip(lower=0)
)

In [26]:
train_store["PromoActive"] = (
    train_store["Promo"] > 0
).astype(int)

In [27]:
train_store["IsHoliday"] = (
    train_store["StateHoliday"] != "0"
).astype(int)

In [28]:
train_store["IsSchoolHoliday"] = (
    train_store["SchoolHoliday"]
).astype(int)

In [29]:
train_store["SalesPerCustomer"] = (
    train_store["Sales"] /
    train_store["Customers"].replace(0, np.nan)
)

train_store["SalesPerCustomer"] = (
    train_store["SalesPerCustomer"]
    .fillna(0)
)

In [30]:
train_store["StoreOpen"] = (
    train_store["Open"]
).astype(int)

In [31]:
def get_season(month):

    if month in [12, 1, 2]:
        return "Winter"

    elif month in [3, 4, 5]:
        return "Spring"

    elif month in [6, 7, 8]:
        return "Summer"

    else:
        return "Autumn"


train_store["Season"] = (
    train_store["Month"]
    .apply(get_season)
)

In [32]:
train_store["MonthStart"] = (
    train_store["Date"]
    .dt.is_month_start
    .astype(int)
)

train_store["MonthEnd"] = (
    train_store["Date"]
    .dt.is_month_end
    .astype(int)
)

In [33]:
train_store[
[
    "CompetitionAge",
    "PromoActive",
    "IsHoliday",
    "SalesPerCustomer",
    "Season"
]
].head()

,CompetitionAge,PromoActive,IsHoliday,SalesPerCustomer,Season
0,7.0,1,0,9.482883,Summer
1,8.0,1,0,9.702400,Summer
2,9.0,1,0,10.126675,Summer
3,6.0,1,0,9.342457,Summer
4,0.0,1,0,8.626118,Summer


In [34]:
train_store.to_csv(
    "../data/processed/rossmann_clean.csv",
    index=False
)

superstore_clean.to_csv(
    "../data/processed/superstore_clean.csv",
    index=False
)

In [35]:
train_store.columns.tolist()

['Store',
 'DayOfWeek',
 'Date',
 'Sales',
 'Customers',
 'Open',
 'Promo',
 'StateHoliday',
 'SchoolHoliday',
 'StoreType',
 'Assortment',
 'CompetitionDistance',
 'CompetitionOpenSinceMonth',
 'CompetitionOpenSinceYear',
 'Promo2',
 'Promo2SinceWeek',
 'Promo2SinceYear',
 'PromoInterval',
 'Year',
 'Month',
 'Day',
 'Week',
 'Quarter',
 'IsWeekend',
 'CompetitionAge',
 'PromoActive',
 'IsHoliday',
 'IsSchoolHoliday',
 'SalesPerCustomer',
 'StoreOpen',
 'Season',
 'MonthStart',
 'MonthEnd']

In [36]:
for col in train_store.columns:
    print(col)

Store
DayOfWeek
Date
Sales
Customers
Open
Promo
StateHoliday
SchoolHoliday
StoreType
Assortment
CompetitionDistance
CompetitionOpenSinceMonth
CompetitionOpenSinceYear
Promo2
Promo2SinceWeek
Promo2SinceYear
PromoInterval
Year
Month
Day
Week
Quarter
IsWeekend
CompetitionAge
PromoActive
IsHoliday
IsSchoolHoliday
SalesPerCustomer
StoreOpen
Season
MonthStart
MonthEnd
